Code by Simona Cariello - INGV Catania


Reference: Cariello, S., Corradino, C., Torrisi, F., & Del Negro, C. (2023). Cascading machine learning to monitor volcanic thermal activity using orbital infrared data: From detection to quantitative evaluation. Remote Sensing, 16(1), 171.

Instructions:



1.   You will need to create a Google and Google Earth Engine (GEE) account if not created yet. Click consense for use of Google services. Type your GEE project in the "gee_name" variable. YOU SHOULD BE LOGGED INTO COLAB WITH THE SAME GOOGLE ACCOUNT YOU SET FOR DRIVE AND GEE
2.   Set the starting and ending date and the volcano

# Libraries

In [ ]:
!pip install geemap
from google.colab import drive
import geemap
import os
import pandas as pd
import glob
from statistics import mode
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import time
import numpy as np
import time
from datetime import datetime, timedelta
from PIL import Image
from torchvision import transforms
from datetime import datetime

# Project_name

In [2]:
import ee

# Mount of the Drive
drive.mount('/content/drive')

#insert your GOOGLE EARTH ENGINE project name!!
gee_project_name = 'ee-simonacariello' #@param {type: "string"}
ee.Authenticate()
ee.Initialize(project=gee_project_name)


#You will use the pretreined model from our asset
assetId = 'projects/victor-corradinoclaudia/assets/RF_TA_app';
means = ee.Image('projects/victor-corradinoclaudia/assets/means_TA_app');
std = ee.Image('projects/victor-corradinoclaudia/assets/std_TA_app');
rf_classifier = ee.Classifier.load(assetId)

Mounted at /content/drive


# Input choices

In [3]:
satellite='Sentinel'

volcano = 'Etna' #@param ["Etna", "Stromboli", "KlyuchevskayaSopka", "Pacaya", "Kilauea", "CumbreVieja", "Shiveluch", "ErtaAle", "Ambrym", "Popo"]

start_date = "2026-01-01" #@param {type: "date"}
#start_hour = "00:12" #@param ["00:12", "00:27", "00:42", "00:57", "01:12", "01:27", "01:42", "01:57", "02:12", "02:27", "02:42", "02:57", "03:12", "03:27", "03:42", "03:57", "04:12", "04:27", "04:42", "04:57", "05:12", "05:27", "05:42", "05:57", "06:12", "06:27", "06:42", "06:57", "07:12", "07:27", "07:42", "07:57", "08:12", "08:27", "08:42", "08:57", "09:12", "09:27", "09:42", "09:57", "10:12", "10:27", "10:42", "10:57", "11:12", "11:27", "11:42", "11:57", "12:12", "12:27", "12:42", "12:57", "13:12", "13:27", "13:42", "13:57", "14:12", "14:27", "14:42", "14:57", "15:12", "15:27", "15:42", "15:57", "16:12", "16:27", "16:42", "16:57", "17:12", "17:27", "17:42", "17:57", "18:12", "18:27", "18:42", "18:57", "19:12", "19:27", "19:42", "19:57", "20:12", "20:27", "20:42", "20:57", "21:12", "21:27", "21:42", "21:57", "22:12", "22:27", "22:42", "22:57", "23:12", "23:27", "23:42", "23:57"]

end_date = "2026-01-12" #@param {type: "date"}
#end_hour = "00:57" #@param ["00:12", "00:27", "00:42", "00:57", "01:12", "01:27", "01:42", "01:57", "02:12", "02:27", "02:42", "02:57", "03:12", "03:27", "03:42", "03:57", "04:12", "04:27", "04:42", "04:57", "05:12", "05:27", "05:42", "05:57", "06:12", "06:27", "06:42", "06:57", "07:12", "07:27", "07:42", "07:57", "08:12", "08:27", "08:42", "08:57", "09:12", "09:27", "09:42", "09:57", "10:12", "10:27", "10:42", "10:57", "11:12", "11:27", "11:42", "11:57", "12:12", "12:27", "12:42", "12:57", "13:12", "13:27", "13:42", "13:57", "14:12", "14:27", "14:42", "14:57", "15:12", "15:27", "15:42", "15:57", "16:12", "16:27", "16:42", "16:57", "17:12", "17:27", "17:42", "17:57", "18:12", "18:27", "18:42", "18:57", "19:12", "19:27", "19:42", "19:57", "20:12", "20:27", "20:42", "20:57", "21:12", "21:27", "21:42", "21:57", "22:12", "22:27", "22:42", "22:57", "23:12", "23:27", "23:42", "23:57"]

vulcano_name=volcano

path_standard= '/content/drive/MyDrive/'#@param {type: "string"}

In [4]:
if volcano =="KlyuchevskayaSopka":
    KlyuchevskayaSopka= ee.Geometry.Rectangle([160.559779548029,56.027360667989434, 160.7070361614637,56.10592827864421])
    roi=KlyuchevskayaSopka

if volcano == "Pacaya":
  Pacaya = ee.Geometry.Rectangle([-90.64344952935497,14.342253003280884,-90.55954955071779,14.422353164487612]);
  roi= Pacaya

if volcano == "Kilauea":
  Kilauea = ee.Geometry.Rectangle([-155.32543968441655,19.365898439973915,-155.24129726501096,19.447879828132617]);
  roi=Kilauea

if volcano == "CumbreVieja":
  CumbreVieja = ee.Geometry.Rectangle([-17.935,28.58,-17.855,28.64]);
  roi= CumbreVieja

if volcano == "Shiveluch":
  Shiveluch = ee.Geometry.Rectangle([161.2498024847712,56.60203161995829,161.40072344744814,56.679734202945546]);
  roi=Shiveluch

if volcano == "ErtaAle":
  ErtaAle = ee.Geometry.Rectangle([40.62002040787817,13.566694707728216,40.70338165426926,13.647126577664434]);
  roi= ErtaAle

if volcano == "Ambrym":
  Ambrym =ee.Geometry.Rectangle([168.11397992729522,-16.287666451684366,168.19888898949438,-16.207890880084385]);
  roi=Ambrym

if volcano == "Popo":
  Popo = ee.Geometry.Rectangle([-98.67055097887622,18.982014651797694,-98.58523040367133,19.062818973202514]);
  roi=Popo

if volcano == "Etna":
  Etna=ee.Geometry.Rectangle([14.943204766569895,37.71062886357165, 15.04490862370018,37.79139299333345]);
  roi=Etna

if volcano == "Stromboli":
  Stromboli=ee.Geometry.Rectangle([15.184154090585974,38.76561861910066,15.24186475961476,38.81237410917109]);
  roi=Stromboli

# RF Functions

In [5]:
import os
import geemap
import ee
from PIL import Image


def tif_to_jpg_rf(tif_path, jpg_path, quality=95):
    img = Image.open(tif_path)
    img = img.convert("RGB")
    img.save(jpg_path, "JPEG", quality=quality)


def export_anomaly_jpgs(
    d,
    roi,
    out_dir,
    scale=20,
    prefix='Etna_Anomaly',
    max_images=None
):
    img_list = d.toList(d.size())
    n_imgs = d.size().getInfo()

    if max_images is not None:
        n_imgs = min(n_imgs, max_images)

    print("Numero immagini da esportare:", n_imgs)

    # DEM + hillshade (una sola volta)
    dem = ee.Image('USGS/SRTMGL1_003').clip(roi)
    hillshade = ee.Terrain.hillshade(dem)

    dem_vis = dem.visualize(
        min=0,
        max=2500,
        palette=['0c2c84', '41b6c4', 'ffffcc']
    )

    hillshade_vis = hillshade.visualize(
        min=100,
        max=255,
        palette=['000000', 'FFFFFF']
    )

    for i in range(n_imgs):

        img = ee.Image(img_list.get(i))

        date_str = (
            ee.Date(img.get('system:time_start'))
            .format('YYYYMMdd')
            .getInfo()
        )

        anomaly = img.select('Map').eq(1)

        # B12–B11–B4 sull’anomalia
        anomaly_vis = (
            img.select(['B12', 'B11', 'B4'])
            .updateMask(anomaly)
            .clip(roi)
            .visualize(min=0, max=4000)
        )

        final_map = ee.ImageCollection([
            dem_vis,
            hillshade_vis,
            anomaly_vis
        ]).mosaic()

        task = ee.batch.Export.image.toDrive(
            image=final_map,
            description=f'{prefix}_{date_str}_{int(time.time())}',
            folder='GEE_exports',
            fileNamePrefix=f'{prefix}_{date_str}',
            region=roi,
            scale=scale,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        time.sleep(120)
        # -----------------------------
        # CONVERSIONE JPG
        # -----------------------------
        tif_to_jpg_rf(tif_path, jpg_path)

        print(f"✔ Salvato: {jpg_path}")


In [6]:
def BandsNormalization(image,bands, roi):
  def prova(name):
    name = ee.String(name);
    band = image.select(bands).select(name);
    return band.unitScale(ee.Number(minMax.get(name.cat("_min"))), ee.Number(minMax.get(name.cat("_max")))).toBands().rename(image.select(bands).bandNames());
  minMax = image.select(bands).reduceRegion(reducer= ee.Reducer.minMax(),geometry=roi,scale= 20, maxPixels= 10e9);
  input = prova(ee.ImageCollection.fromImages(image.select(bands).bandNames()))
  return input

In [7]:
def CNESf(image):
  timestamp = image.get("system:time_start")
  JD= (ee.Number(timestamp).divide(86400000)).add(2440587.5)
  MJD = ee.Number(JD).subtract(2400000.5)
  CNES = (ee.Number(MJD).subtract(33282)).floor()
  return CNES


In [8]:
def S2(image):
  bands_s2=["B2", "B3", "B4","B8A", "B11", "B12"];
  col=image.select(bands_s2).addBands(image.select("B2").rename("Band1")).addBands(image.select("B3").rename("Band2")).addBands(image.select("B4").rename("Band3")).addBands(image.select("B8A").rename("Band4")).addBands(image.select("B11").rename("Band5")).addBands(image.select("B12").rename("Band6"))
  return col

In [9]:
def d_tf(CNES):
 t= ee.Number(CNES)
 t1= t.subtract(ee.Number(2))
 c1=ee.Number(0.0172).multiply(t1)
 cos= ee.Number(c1).cos()

 c2 = ee.Number(0.01673).multiply(cos)
 c3= ee.Number(1).subtract(c2)
 p=c3.pow(ee.Number(2))
 d_t= ee.Number(1).divide(p)
 return d_t

In [10]:
def DN_to_Reflectance_to_Radiance(image,year):

 #if year == 2022:
        #return image==image.subtract(10000)

 CNES = CNESf(ee.Image(image))

 d_t = d_tf(CNES)

 SOLAR_IRRADIANCE_B1 = image.get('SOLAR_IRRADIANCE_B1')
 MEAN_INCIDENCE_ZENITH_ANGLE_B1 = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B1')

 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B1;

 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));

 cos_z_angle_radians = ee.Number(z_angle_radians).cos()


 B1_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B1'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B1),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });


 SOLAR_IRRADIANCE_B2 = image.get('SOLAR_IRRADIANCE_B2')
 MEAN_INCIDENCE_ZENITH_ANGLE_B2 = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B2')
 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B2;

 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));

 cos_z_angle_radians = ee.Number(z_angle_radians).cos()

 B2_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B2'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B2),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });

 SOLAR_IRRADIANCE_B3 = image.get('SOLAR_IRRADIANCE_B3')
 MEAN_INCIDENCE_ZENITH_ANGLE_B3 = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B3')


 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B3;

 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));

 cos_z_angle_radians = ee.Number(z_angle_radians).cos()

 B3_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B3'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B3),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });

 SOLAR_IRRADIANCE_B4 = image.get('SOLAR_IRRADIANCE_B4')
 MEAN_INCIDENCE_ZENITH_ANGLE_B4 = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B4')
 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B4;

 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));

 cos_z_angle_radians = ee.Number(z_angle_radians).cos()

 B4_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B4'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B4),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });

 SOLAR_IRRADIANCE_B5 = image.get('SOLAR_IRRADIANCE_B5')
 MEAN_INCIDENCE_ZENITH_ANGLE_B5 = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B5')
 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B5;
 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));
 cos_z_angle_radians = ee.Number(z_angle_radians).cos()

 B5_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B5'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B5),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });

 SOLAR_IRRADIANCE_B6 = image.get('SOLAR_IRRADIANCE_B6')
 MEAN_INCIDENCE_ZENITH_ANGLE_B6 = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B6')
 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B6;
 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));

 cos_z_angle_radians = ee.Number(z_angle_radians).cos()

 B6_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B6'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B6),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });

 SOLAR_IRRADIANCE_B7 = image.get('SOLAR_IRRADIANCE_B7')
 MEAN_INCIDENCE_ZENITH_ANGLE_B7 = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B7')
 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B7;

 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));

 cos_z_angle_radians = ee.Number(z_angle_radians).cos()

 B7_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B7'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B7),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });

 SOLAR_IRRADIANCE_B8 = image.get('SOLAR_IRRADIANCE_B8')
 MEAN_INCIDENCE_ZENITH_ANGLE_B8 = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B8')
 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B8;
 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));
 cos_z_angle_radians = ee.Number(z_angle_radians).cos()

 B8_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B8'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B8),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });

 SOLAR_IRRADIANCE_B8A = image.get('SOLAR_IRRADIANCE_B8A')
 MEAN_INCIDENCE_ZENITH_ANGLE_B8A = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B8A')

 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B8A;

 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));

 cos_z_angle_radians = ee.Number(z_angle_radians).cos()

 B8A_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B8A'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B8A),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });

 SOLAR_IRRADIANCE_B9 = image.get('SOLAR_IRRADIANCE_B9')
 MEAN_INCIDENCE_ZENITH_ANGLE_B9 = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B9')
 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B9;
 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));

 cos_z_angle_radians = ee.Number(z_angle_radians).cos()

 B9_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B9'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B9),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });
 SOLAR_IRRADIANCE_B10 = image.get('SOLAR_IRRADIANCE_B10')
 MEAN_INCIDENCE_ZENITH_ANGLE_B10 = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B10')
 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B10;

 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));

 cos_z_angle_radians = ee.Number(z_angle_radians).cos()

 B10_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B10'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B10),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });

 SOLAR_IRRADIANCE_B11 = image.get('SOLAR_IRRADIANCE_B11')
 MEAN_INCIDENCE_ZENITH_ANGLE_B11 = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B11')
 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B11;

 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));

 cos_z_angle_radians = ee.Number(z_angle_radians).cos()

 B11_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B11'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B11),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });

 SOLAR_IRRADIANCE_B12 = image.get('SOLAR_IRRADIANCE_B12')
 MEAN_INCIDENCE_ZENITH_ANGLE_B12 = image.get('MEAN_INCIDENCE_ZENITH_ANGLE_B12')
 z_angle_degree = MEAN_INCIDENCE_ZENITH_ANGLE_B12;

 z_angle_radians = ee.Number(z_angle_degree).multiply(ee.Number.expression('Math.PI').divide(180));

 cos_z_angle_radians = ee.Number(z_angle_radians).cos()

 B12_radiance = image.expression(
    ' ( (img/QUANTIFICATION_VALUE) * si * cos_z * d_t ) / (6.283185307179586 )',
    {
      'img':image.select('B12'),
      'QUANTIFICATION_VALUE': 10000,
      'si': ee.Number(SOLAR_IRRADIANCE_B12),
      'cos_z': cos_z_angle_radians,
      'd_t': d_t
    });

 image = ee.Image(image).addBands(B1_radiance.rename('B1_radiance')).addBands(B2_radiance.rename('B2_radiance')).addBands(B3_radiance.rename('B3_radiance')).addBands(B4_radiance.rename('B4_radiance')).addBands(B5_radiance.rename('B5_radiance')).addBands(B6_radiance.rename('B6_radiance')).addBands(B7_radiance.rename('B7_radiance')).addBands(B8_radiance.rename('B8_radiance')).addBands(B8A_radiance.rename('B8A_radiance')).addBands(B9_radiance.rename('B9_radiance')).addBands(B10_radiance.rename('B10_radiance')).addBands(B11_radiance.rename('B11_radiance')).addBands(B12_radiance.rename('B12_radiance')).select('B1_radiance', 'B2_radiance', 'B3_radiance', 'B4_radiance', 'B5_radiance', 'B6_radiance', 'B7_radiance', 'B8_radiance', 'B8A_radiance', 'B9_radiance', 'B10_radiance', 'B11_radiance', 'B12_radiance').rename('B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12')
 return image

In [11]:
#You can activate it if you want a hotspot part like output

def generateComp2(sst0, comp, geo, res, vec, nc):

    # === campionamento per clustering ===
    training = (
        sst0
        .select(['B8A', 'B11'])
        .multiply(comp)
        .sample(
            region=vec,
            scale=res,
            numPixels=1e8
        )
    )

In [12]:
def performMap(feature):

  simple = feature.simplify(10000);
  center = simple.centroid(100);
  l=feature.geometry().convexHull(10).length()
  return (
    feature.set('length',l)
    )

In [13]:
NEcrater =ee.Geometry.Polygon(
        [[[14.995063426876504, 37.75583930060303],
          [14.994934680843789, 37.7553303482914],
          [14.994805934811074, 37.75431243316457],
          [14.996093395138223, 37.7536338152996],
          [14.996737126984492, 37.753175747373255],
          [14.997724178219277, 37.75332843522972],
          [14.998367908382852, 37.75407491761984],
          [14.998968723202188, 37.755024975223854],
          [14.998496654415566, 37.75641610899025],
          [14.997123363399941, 37.75668754667521],
          [14.995707157040078, 37.75631431960167]]])

SEcrater = ee.Geometry.Polygon(
        [[[14.99742377080961, 37.746847294321555],
          [14.998196247005898, 37.74582926249287],
          [14.99971973813523, 37.745591729888055],
          [15.001329067135293, 37.746168607991734],
          [15.001929881954629, 37.747695643481485],
          [15.001071575069863, 37.74908691505127],
          [14.999526622677285, 37.749561984878106],
          [14.99841082372709, 37.74908691505127],
          [14.997209194088418, 37.748068914030995]]])

Voragine =ee.Geometry.Polygon(
        [[[14.994119289303262, 37.75295519121039],
          [14.993990543270547, 37.751258603756405],
          [14.995320918941934, 37.7509192815975],
          [14.996780040646035, 37.75098714615377],
          [14.997295024776895, 37.752039038816754],
          [14.99665129461332, 37.752887328459124],
          [14.994977596188027, 37.75329450403302]]])

BoccaNuova =ee.Geometry.Polygon(
        [[[14.993775966549356, 37.75125860375643],
          [14.99240267553373, 37.75234442420836],
          [14.99090063848539, 37.75207297058918],
          [14.990771892452676, 37.75034243035625],
          [14.991286876583535, 37.748747582933845],
          [14.99441969671293, 37.748204648309496],
          [14.995621326351602, 37.74915478128805],
          [14.994484082845556, 37.750749616054264]]])


In [14]:
def map_rf_v2(sst0):


  res=20;
  bands=["Band1","Band2","Band3","Band4","Band5","Band6"]
  geo=roi
  # clip
  sst0 = sst0.clip(roi)

  # differenza temporale
  dif = sst0.date().difference(ee.Date('2022-01-23'), 'day')

  image_old  = DN_to_Reflectance_to_Radiance(sst0, 2021)
  image_2022 = DN_to_Reflectance_to_Radiance(sst0, 2022)

  # anno immagine
  year = ee.Date(sst0.get('system:time_start')).get('year')

  # DN → Radiance (condizionale)
  image = ee.Image(
      ee.Algorithms.If(
          dif.lt(0),
          DN_to_Reflectance_to_Radiance(sst0, 2021),
          DN_to_Reflectance_to_Radiance(sst0, 2022)
      )
  )

  # S2 VIS–NIR–SWIR
  sst = S2(image).double()

  # selezione bande RF
  composite = sst.select(bands).clip(roi)

  # normalizzazione
  test = composite.subtract(means).divide(std)

  # classificazione
  s2km = test.classify(rf_classifier)

  # maschera
  comp = s2km.updateMask(s2km)

  # vettorializzazione
  classes = comp.reduceToVectors(
      reducer=ee.Reducer.countEvery(),
      geometry=roi,
      scale=20,
      maxPixels=1e10,
      bestEffort=True
  )
  vec = ee.FeatureCollection(classes)

  # numero pixel
  np = ee.Number(
      comp.reduceRegion(
          ee.Reducer.sum(),
          geo,
          20,
          maxPixels=1e10
      ).values().get(0)
  ).toShort()

  # comp2 (LVQ)
  #comp2 = generateComp2(sst0, comp, geo, 20, vec, 2)

  # cast
  comp_u = comp.toUint16().rename('Map')
  #comp2_u = comp2.toUint16().rename('Map_red')


  def sum_region(img, geom, scale):
    return (
        img.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geom,
            scale=scale,
            bestEffort=True
        )
        .values()
        .get(0)
    )

  ne = ee.Number(sum_region(comp, NEcrater, res))
  se = ee.Number(sum_region(comp, SEcrater, res))
  vor = ee.Number(sum_region(comp, Voragine, res))
  bn = ee.Number(sum_region(comp, BoccaNuova, res))



    # numero feature
  n_feat = vec.size()

  # geometria SOLO se esiste
  geom = ee.Geometry(
      ee.Algorithms.If(
          n_feat.gt(0),
          vec.sort('count', False).first().geometry().convexHull(10),
          roi   # fallback: ROI
      )
  )

  length = ee.Number(
      ee.Algorithms.If(
          n_feat.gt(0),
          geom.perimeter().divide(2),
          0
      )
  )

  elevation = ee.Image("USGS/SRTMGL1_003").select('elevation')

  altitude = ee.Number(
    ee.Algorithms.If(
        n_feat.gt(0),
        elevation
        .sample(
            region=geom.centroid(),
            scale=30,
            geometries=False
        )
        .first()
        .get('elevation'),
        0
    )
)



  return (
  sst0
  .addBands(comp_u)
  .set('NumPix_Anomaly', np)
  .set('Area_Anomaly', np.multiply(400))
  .set('NumPix_Hotspot', 0)
  .set('Area_Hotspot', 0)
  .set('NE',ne)
  .set('SE',se)
  .set('BN',bn)
  .set('VOR',vor)
  .set('Length', length)
  .set('Altitude', altitude)
  .copyProperties(sst0, sst0.propertyNames())
)

In [15]:
def image_to_feature(img):
    return ee.Feature(
        None,
        img.toDictionary([
            'system:time_start',
            'NumPix_Anomaly',
            'Area_Anomaly',
            'NumPix_Hotspot',
            'Area_Hotspot'
            'NE',
            'SE',
            'BN',
            'VOR',
            'Length',
            'Altitude'
        ])
    )

def random_forest(colS2_rf):
    d = colS2_rf.map(map_rf_v2)
    fc = ee.FeatureCollection(d.map(image_to_feature))
    return d, fc


# SqueezeNet Function

In [16]:
def tif_to_jpg(vulcano_name, satellite, coll_list):
  #Script Original with band [8,11,12]
  print("I’m starting to convert images to jpg")
  i=0
  from osgeo import gdal
  path=  str(path_standard) +str(vulcano_name) + '/tif'
  path_jpg=  str(path_standard)+str(vulcano_name) + '/JPG'

  coll_list_tif=list()
  for el in coll_list:
    coll_list_tif.append(el + '.tif')

  #for filename in os.listdir(path)[-contatore_img:]:
  for filename in os.listdir(path):
    if filename in coll_list_tif:
      f = os.path.join(path, filename)
      #band=['12' ,'11','8']
      options_list = [
        '-ot Byte',
        '-of JPEG',
        '-b 1',
        '-b 2',
        '-b 3',
        '-scale',
        '-outsize 224 224',
      ]
      options_string = " ".join(options_list)

      if satellite=='Sentinel':
        gdal.Translate(
          str(path_standard) +str(vulcano_name) + '/JPG/' +str(filename[0:38])+'.jpg',
          f,
          options=options_string
          )
      print("Waiting time between multiple images")
      time.sleep(10)
      i=i+1

In [17]:
def run_sentinel_NEW(roi):
  #Sentinel 2
  coll_list=list()
  indice=list()
  satellite='Sentinel'
  data_oggi=datetime.now().strftime('%Y-%m-%d') #if end_date is now

  colS2=ee.ImageCollection("COPERNICUS/S2_HARMONIZED").filterDate(start_date,end_date).filterBounds(roi)


  def red(image):
    np = ee.Number(image.reduceRegion(
        reducer= ee.Reducer.count(),
        geometry= roi,
        scale= 20,
        maxPixels= 1e9
    ).values().get(0))
    return image.set("dim", np)

  newcollection= colS2.map(red)

  def add_day(image):
    return image.set(
        'day', image.date().format('YYYY-MM-dd')
    )

  newcollection = newcollection.map(add_day)

  days = newcollection.distinct(['day'])

  def one_image_per_day(image):
      day = image.get('day')
      daily = newcollection.filter(ee.Filter.eq('day', day))
      return daily.sort('dim', False).first()

  newcollection = ee.ImageCollection(
      days.map(one_image_per_day)
  )



  coll_export=newcollection.aggregate_array('dim').getInfo()
  #print(coll_export)
  time.sleep(5)
  conta= newcollection.filterMetadata('dim','greater_than',60000)
  coll_list= conta.aggregate_array('system:index').getInfo()
  print("List of images found: " + str(coll_list))
  count=conta.size()
  count=str(count.getInfo())
  print("I found: " + count + " images")
  count=int(count)

  if volcano == "Etna":
    newcollection = newcollection.filter(ee.Filter.stringEndsWith("system:index", "VB"))


  if count==0:
    print("There are no new images")
  else:
    out_dir =str(path_standard)+ str(volcano) + "/tif"
    colS2_rf = newcollection.filterMetadata('dim','greater_than',60000)
    colS2 = newcollection.filterMetadata('dim','greater_than',60000).select(["B12","B11","B8"]) #modificate
    geemap.ee_export_image_collection(colS2, out_dir=out_dir, scale=20, region=roi)

    print("I downloaded the images, now I convert them, wait")
    time.sleep(60)
    directory_new= str(path_standard) + str(vulcano_name) + "/JPG"
    os.makedirs(directory_new)
    time.sleep(2)



    print("I'm starting the Random Forest algorithm")
    d, metrics = random_forest(colS2_rf)

    #You can activate if you want to export the map
    """
    d, metrics = random_forest(colS2_rf)
    export_anomaly_jpgs(
    d=d,
    roi=roi,
    out_dir="/content/drive/MyDrive/",
    scale=20,
    prefix="Etna_Anomaly",
    max_images=10,     # opzionale
    #delete_tif=True    # opzionale
    )

"""



    data = metrics.getInfo()

    NumPix_Anomaly = []
    Area_Anomaly = []
    NumPix_Hotspot = []
    Area_Hotspot = []
    NE = []
    SE = []
    BN = []
    VOR = []
    Length = []
    Altitude = []

    for f in data['features']:
        p = f['properties']

        NumPix_Anomaly.append(p.get('NumPix_Anomaly'))
        Area_Anomaly.append(p.get('Area_Anomaly'))
        NumPix_Hotspot.append(p.get('NumPix_Hotspot'))
        Area_Hotspot.append(p.get('Area_Hotspot'))
        NE.append(p.get('NE'))
        SE.append(p.get('SE'))
        BN.append(p.get('BN'))
        VOR.append(p.get('VOR'))
        Length.append(p.get('Length'))
        Altitude.append(p.get('Altitude'))

    #print(NumPix_Anomaly)
    #print(Area_Anomaly)
    #print(NumPix_Hotspot)
    #print(Area_Hotspot)
    #print(NE)
    #print(SE)
    #print(BN)
    #print(VOR)
    #print(Length)
    #print(Altitude)


    tif_to_jpg(volcano, satellite, coll_list)


    #time.sleep(60)
    #train_model(volcano, table_model_stromboli, satellite, coll_list)
    rows = []

    for f in data['features']:
        p = f['properties']

        rows.append({
            'Date': ee.Date(p['system:time_start'])
                    .format('YYYY-MM-dd HH:mm')
                    .getInfo(),
            'Flag': None,
            'Activity': None,
            'Prediction': None,
            'NumPix_Anomaly': p.get('NumPix_Anomaly'),
            'Area_Anomaly': p.get('Area_Anomaly'),
            'NumPix_Hotspot': p.get('NumPix_Hotspot'),
            'Area_Hotspot': p.get('Area_Hotspot'),
            'Length_m': p.get('Length'),
            'Altitude_m': p.get('Altitude'),
            'NE': p.get('NE'),
            'SE': p.get('SE'),
            'BN': p.get('BN'),
            'VOR': p.get('VOR'),
            'Sensor': 'MSI-OLI',
            'Satellite': 'Sentinel 2',
            #cascading_output
            #path_img
            #path_img_ML_A
            #path_img_ML_H

        })

    df = pd.DataFrame(rows)

  return df



In [18]:
def train_model_new(df, satellite):
  print("I'm starting the classification")


  path_jpg= str(path_standard) + str(vulcano_name) + '/JPG'

  #Delete different extension
  for filename in os.listdir(path_jpg):
    if filename.endswith(".xml"):
        os.remove(os.path.join(path_jpg, filename))

  time.sleep(10)

  print("Time finished, starting to open images")

  df['Date'] = pd.to_datetime(df['Date']).dt.date

  #print(df['Date'])

  for filename in os.listdir(path_jpg):
    input_image = Image.open(str(path_standard) + str(vulcano_name) + '/JPG/'+ filename).convert('RGB')
    input_image.show()

    # Preprocessing of the data

    preprocess = transforms.Compose([
        #transforms.Resize(256),
        #transforms.CenterCrop(224),
        #transforms.RandomHorizontalFlip(),
        #transforms.RandomVerticalFlip(),
        #transforms.Pad(padding=10, fill=0, padding_mode='edge'),
        #transforms.RandomCrop(224),
        transforms.ToTensor(),  #necessary for the process
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), #normalization parameters taken from the values used for the original training of the model (from Torchvision Documentation)
    ])
    input_tensor = preprocess(input_image)
    input_batch = input_tensor.unsqueeze(0) # create a mini-batch as expected by the model



    # Load the trained models
    predizioni = np.empty(11, dtype = int)
    import torch
    i=0
    models=['model1','model2','model3','model4','model5','model6','model7','model8','model9','model10', 'model11']
    for f in models:
      #print(f)
      path_models=str(path_standard)
      #PATH=path_models+'models_extend/'+ f + '.pt'
      PATH=path_models+'models_s/'+ f + '.pt'
      f = torch.load(PATH, weights_only=False)
      #print(f)
      if torch.cuda.is_available():
        input_batch = input_batch.to('cuda')
        f.to('cuda')

      with torch.no_grad():
        output = f(input_batch) # Tensor of shape 2, with confidence scores
        #print(output)
        # The output has unnormalized scores. To get probabilities, it is possible to use the softmax function
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        preds = output.to('cpu').max(1)[1].numpy()
        preds=np.argmax(output).numpy()
        #print(preds)
        # Prediction output
        predictions = []
        #predictions.extend(list(preds))
        #print(predictions)
        #print('Prediction_: ' + str(i) , predictions)
        predizioni[i]=preds
        i=i+1
    #print(predizioni)

    #predizionis=predizioni
    activity=''
    ensemble_prediction=mode(predizioni)
    #print(ensemble_prediction)

    if ensemble_prediction==0:
      ensemble_prediction=1
      activity= 'Thermal anomalies'
    elif ensemble_prediction==1:
      ensemble_prediction=2
      activity= 'Lava flows'
    elif ensemble_prediction==2:
      ensemble_prediction=3
      activity= 'No activity'
    elif ensemble_prediction==3:
      ensemble_prediction=-1
      activity= 'Cloudy'

    if satellite=='Sentinel':
      #Sentinel 2

      date_str = (
      filename[0:4] + '-' +
      filename[4:6] + '-' +
      filename[6:8]
      #filename[9:11] + ':' +
      #filename[11:13]
      )
      print(date_str)
      date_dt = pd.to_datetime(date_str, format='%Y-%m-%d').date()

      mask = df['Date'] == date_dt

      #print("mask", mask)
      #time.sleep(10)


      #print(ensemble_prediction)
      #print(activity)
      #print(str(predizioni))

      df.loc[mask, 'Flag'] = ensemble_prediction
      df.loc[mask, 'Activity'] = activity
      df.loc[mask, 'Prediction'] = str(predizioni)

  print("Writing on Excel file")

  df.to_excel(
    str(path_standard) + str(vulcano_name) + f"/{vulcano_name}.xlsx",
    index=False
    )



In [19]:
def tif_to_jpg_NEW(vulcano_name, satellite):
  print("I'm starting to convert the images")
  i=0
  from osgeo import gdal
  path= str(path_standard) +str(vulcano_name) + '/tif'
  path_jpg= str(path_standard) +str(vulcano_name) + '/JPG'


  #for filename in os.listdir(path)[-contatore_img:]:
  for filename in os.listdir(path):

    f = os.path.join(path, filename)
    #print(f)
    #band=['12' ,'11','8']
    options_list = [
      '-ot Byte',
      '-of JPEG',
      '-b 1',
      '-b 2',
      '-b 3',
      '-scale',
      '-outsize 224 224',
    ]
    options_string = " ".join(options_list)


    if satellite=='Sentinel':
      gdal.Translate(
        str(path_standard) +str(vulcano_name) + '/JPG/' +str(filename[0:38])+'.jpg',
        f,
        options=options_string
        )
    print("Wait..")
    time.sleep(5)
    i=i+1

In [20]:
def temporal_series():
  table_model = pd.read_excel(str(path_standard)+ str(vulcano_name) + "/" +str(vulcano_name)+".xlsx")
  ax2 = table_model.plot.scatter(x='Date', y='Activity', c='red',figsize=(26, 10))
  row=(np.array(table_model['Date']))
  max_len=len(row)
  #print(max_len)
  xticks=np.arange(0,max_len,10)
  xlabels=(row[xticks])
  #plt.ylim(top='Lava flows',bottom='Thermal anomalies')  # return the current ylim
  start, end = ax2.get_xlim()
  ax2.xaxis.set_ticks(np.arange(start, end, 100))
  plt.show()

In [21]:
import pandas as pd
import numpy as np

def cascading_output(
    excel_path,
    area_col='Area_Anomaly',
    pred_col='Prediction',
    n_models=11,
    n_classes=4,
    threshold=0.85,
    save=True
):
    """
    Calcola:
    - Probabilità per classe
    - Max_Prob
    - Decisione finale a cascata

    Parametri:
    - excel_path: path al file Excel
    - area_col: nome colonna Area_Anomaly
    - pred_col: nome colonna Prediction
    - n_models: numero modelli ensemble
    - n_classes: numero classi
    - threshold: soglia di confidenza
    - save: se True, salva l'Excel

    Ritorna:
    - df aggiornato
    """


    df = pd.read_excel(excel_path)


    def compute_probabilities(pred_str):
        preds = np.array(
            [int(x) for x in pred_str.strip('[]').split()]
        )

        probs = np.zeros(n_classes)
        for c in range(n_classes):
            probs[c] = np.sum(preds == c) / n_models

        max_class = int(np.argmax(probs))
        max_prob = float(probs[max_class])

        return probs, max_class, max_prob


    for c in range(n_classes):
        df[f'Prob_{c}'] = np.nan

    df['Max_Class'] = np.nan
    df['Max_Prob'] = np.nan


    for idx, row in df.iterrows():

        if pd.isna(row[pred_col]):
            continue

        probs, max_class, max_prob = compute_probabilities(row[pred_col])

        for c in range(n_classes):
            df.loc[idx, f'Prob_{c}'] = probs[c]

        df.loc[idx, 'Max_Class'] = max_class
        df.loc[idx, 'Max_Prob'] = max_prob


    def final_decision(row):
        probs = {c: row[f'Prob_{c}'] for c in range(n_classes)}
        max_prob = max(probs.values())

        # caso alta confidenza
        if max_prob >= threshold:
            return max(probs, key=probs.get)

        # caso bassa confidenza
        if row[area_col] != 0:
          subset = {k: probs[k] for k in [0, 1]}  # Thermal vs Lava
        else:
          subset = {k: probs[k] for k in [2, 3]}  # No activity vs Cloudy

        return max(subset, key=subset.get)

    df['Final_Class'] = df.apply(final_decision, axis=1)


    class_map = {
        0: 'Thermal anomalies',
        1: 'Lava flows',
        2: 'No activity',
        3: 'Cloudy'
    }

    df['Final_Activity'] = df['Final_Class'].map(class_map)


    if save:
        df.to_excel(excel_path, index=False)

    return df


# RUN Code


In [ ]:
df = run_sentinel_NEW(roi)
time.sleep(60)
train_model_new(df, satellite)
#temporal_series()
df_def = cascading_output(excel_path=str(path_standard) + str(vulcano_name) + f"/{vulcano_name}.xlsx")


# Try the RF and plot the map

In [ ]:
Etna=ee.Geometry.Rectangle([14.943204766569895,37.71062886357165, 15.04490862370018,37.79139299333345]);
#Etna = ee.Geometry.Rectangle([14.965112983701191, 37.71041627444679, 15.067079841611347, 37.79158221832303]) #Etna january 2026

roi=Etna
colS2=ee.ImageCollection("COPERNICUS/S2_HARMONIZED").filterDate(start_date,end_date).filterBounds(roi)
colS2 = colS2.filter(ee.Filter.stringEndsWith("system:index", "VB"))
reference=colS2

etna=ee.Geometry.Point(14.996,37.754);
stromboli=ee.Geometry.Point(15.211912,38.792513)
value=etna
geometry=value
bands=["Band1","Band2","Band3","Band4","Band5","Band6"]
res=20;
radius=8000;

initialPoint = value;

#Initialize with a test point.

geo=initialPoint.buffer(radius);

start ="2026-01-01"
now = "2026-01-08"
end = "2026-01-08"

start_rf=ee.Date(now).advance(-2,"month")
dateRange = ee.DateRange(start, end)

sentinel2=ee.ImageCollection("COPERNICUS/S2_HARMONIZED").filterBounds(initialPoint)#.filterMetadata("CLOUDY_PIXEL_PERCENTAGE", "less_than",50)//.map(cloud_mask_s2);
print(type(sentinel2))
region =geo
sentinel2_SR=sentinel2;
collection=sentinel2.sort("system:time_start");
bands_chart=bands
reference=ee.ImageCollection(collection.filterDate(start, end));

d = reference.map(map_rf_v2)

<class 'ee.imagecollection.ImageCollection'>


In [ ]:
#Map

import geemap
Map = geemap.Map()

img = ee.Image(d.sort('system:time_start', False).first())


date = ee.Date("2025-08-31")

img = ee.Image(
    d.filterDate(date, date.advance(1, 'day')).first()
)

anomaly = img.select('Map').eq(1)


dem = ee.Image('USGS/SRTMGL1_003').clip(roi)


hillshade = ee.Terrain.hillshade(dem)


false_rgb_anomaly = (
    img.select(['B12', 'B11', 'B4'])
    .updateMask(anomaly)
    .clip(roi)
)



Map = geemap.Map()
Map.centerObject(roi, 12)

Map.addLayer(
    dem,
    {
        'min': 0,
        'max': 2500,
        'palette': ['0c2c84', '41b6c4', 'ffffcc']
    },
    'DEM color'
)

Map.addLayer(
    hillshade,
    {
        'min': 100,
        'max': 255,
        'opacity': 0.6
    },
    'Hillshade'
)

Map.addLayer(
    false_rgb_anomaly,
    {
        'min': 0,
        'max': 4000
    },
    'B12-B11-B4 anomaly'
)

Map

Map(center=[37.75101450638294, 14.994056695135807], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
#Save .tif like batch task

# DEM colorato (RGB)
dem_vis = dem.visualize(
    min=0,
    max=2500,
    palette=['0c2c84', '41b6c4', 'ffffcc']
)

# Hillshade RGB (FIX!)
hillshade_vis = hillshade.visualize(
    min=100,
    max=255,
    palette=['000000', 'FFFFFF']
)

# Anomalia RGB
anomaly_vis = false_rgb_anomaly.visualize(
    min=0,
    max=4000
)

# Immagine finale (ORA FUNZIONA)
final_map = ee.ImageCollection([
    dem_vis,
    hillshade_vis,
    anomaly_vis
]).mosaic()

b12_b11_b8 = (
    img.select(['B12', 'B11', 'B8'])
    .clip(roi).visualize(
    min=0,
    max=4000,
    gamma=1.2
    )
)
out_tif = '/content/drive/MyDrive/GEE_exports/2025_08_31_sat.tif'

geemap.ee_export_image(
    b12_b11_b8,
    out_tif,
    scale=20,
    region=roi,
    crs='EPSG:4326',
    file_per_band=False,
)



task = ee.batch.Export.image.toDrive(
    image=final_map,
    description='2025_08_31',
    folder='GEE_exports',
    fileNamePrefix='2025_08_31',
    region=roi,
    scale=20,
    crs='EPSG:4326',
    maxPixels=1e13
)

task.start()
print("🚀 Export avviato correttamente")

task.start()
print("Export avviato su Google Drive")

import time

while task.active():
    print("⏳ Export in corso...")
    time.sleep(30)

print("✅ Export completato")

#even if from id error, don't be afraid, the task has started, you need to change the description of each one.

In [ ]:
#RUN only to delete the previous task
"""
for t in ee.batch.Task.list():
    if t.status()['state'] in ['READY', 'RUNNING', 'FAILED']:
        t.cancel()
        print("Task cancellato:", t.status()['description'])
"""

In [ ]:
tasks = ee.batch.Task.list()
for t in tasks:
    print(t.status())


In [ ]:
from PIL import Image

def tif_to_jpg(tif_path, jpg_path, quality=95):
    img = Image.open(tif_path)
    img = img.convert("RGB")
    img.save(jpg_path, "JPEG", quality=quality)


tif_path = '/content/drive/MyDrive/GEE_exports/yyyy_mm_dd.tif'
jpg_path = "/content/drive/MyDrive/GEE_exports/yyyy_mm_dd.jpg"

tif_to_jpg(tif_path, jpg_path)

print("Conversione completata")